In [0]:
create schema if not exists 03_gold_catalog.obt;
create or replace table 03_gold_catalog.obt.one_big_table as
select 
  o.opportunity_id,
  o.start_date,
  o.end_date,
  o.contract_term,
  o.revenue_amount,
  o.close_status,
  c.customer_id,
  c.customer_name,
  c.country_code,
  c.is_active as is_active_customer,
  c.account_created_date,
  c.industry_type,
  e.employee_id,
  e.employee_name,
  e.role,
  e.region,
  e.hire_date,
  e.is_active as is_active_employee,
  e.last_update,
  p.product_id,
  p.product_name,
  p.plan_name,
  p.billing_cycle,
  p.list_price,
  p.is_active as is_active_product,
  p.created_date as created_date_product,
  er.currency_code,
  er.fx_rate_to_gbp,
  er.effective_date,
  cm.country_name
from 03_gold_catalog.facts.fact_opportunity o

inner join 03_gold_catalog.dimensions.dim_customer c
on o.customer_sk = c.customer_sk

inner join 03_gold_catalog.dimensions.dim_employee e
on o.employee_sk = e.employee_sk

inner join 03_gold_catalog.dimensions.dim_product p
on o.product_sk = p.product_sk

inner join 03_gold_catalog.dimensions.dim_country_master cm
on c.country_code = cm.country_code

inner join 03_gold_catalog.dimensions.dim_exchange_rate er
on cm.currency_code = er.currency_code

In [0]:
create schema if not exists 03_gold_catalog.kpis_from_obt

In [0]:
-- 1
create or replace table `03_gold_catalog`.kpis_from_obt.kpi1 as
with cte1 as(
  select
    customer_id,
    start_date,
    end_date,
    lag (start_date) over (partition by customer_id order by start_date) as prev_start_date,
    lead (start_date) over (partition by customer_id order by start_date) as next_start_date
  from `03_gold_catalog`.obt.one_big_table 
  where close_status='Won'
),
cte2 as (
  select
    count(distinct customer_id) as customers_gained
  from cte1
  where year(start_date) = 2024 and prev_start_date is null
),
cte3 as(
  select 
    count(distinct customer_id) as customers_lost
  from cte1
  where year(end_date) = 2024 and next_start_date is null
)
select 
  customers_gained,
  customers_lost
from cte2 cross join cte3

In [0]:
-- 2
create or replace table `03_gold_catalog`.kpis_from_obt.kpi2 as
with cte1 as(
  select
    customer_id,
    year(end_date) as year,
    lead(start_date) over (partition by customer_id order by start_date) as next_start_date
  from `03_gold_catalog`.obt.one_big_table
  where close_status='Won'
  order by customer_id,start_date
),
cte2 as (
  select *
  from cte1
  where next_start_date is null
),
cte3 as(
  select 
    year,
    count(distinct customer_id) as customers_lost
  from cte2
  group by year
  order by year
),
cte4 as (
  select * , lag(customers_lost) over (order by year) as prev_year_customer_lost
  from cte3
)
select 
  year,
  customers_lost,
  case when customers_lost>prev_year_customer_lost then "Yes" else "No" end as is_customers_lost_increases
from cte4;

In [0]:
-- 3
create or replace table `03_gold_catalog`.kpis_from_obt.kpi3 as
with cte1 as(
  select
    customer_id,
    customer_name,
    date_trunc('month',start_date) as start_month,
    case 
      when contract_term = 'Yearly' then 12
      else 1
    end as no_of_months,
    case 
      when contract_term = 'Yearly' then revenue_amount/12
      else revenue_amount
    end as monthly_revenue
  from `03_gold_catalog`.obt.one_big_table
  where close_status='Won'
  order by customer_id,start_date
),
cte2 as(
  select customer_id,customer_name,
  monthly_revenue,
  explode(sequence(start_month,add_months(start_month, no_of_months - 1), interval 1 month)) as report_month
  from cte1
),
cte3 as (
  select customer_id,customer_name,
  sum(monthly_revenue) as monthly_revenue,
  report_month
  from cte2
  group by customer_id,customer_name,report_month
),
cte4 as (
  select
    customer_id,customer_name,
    count(distinct report_month) as no_of_active_months,
    avg(monthly_revenue) as avg_monthly_revenue,
    sum(monthly_revenue) as total_revenue
  from cte3 
  group by customer_id,customer_name
)
select 
  customer_id, 
  customer_name,
  no_of_active_months,
  total_revenue,
  avg_monthly_revenue
from cte4 
order by total_revenue desc, avg_monthly_revenue desc
limit 15


In [0]:
-- 4
create or replace table `03_gold_catalog`.kpis_from_obt.kpi4 as
with cte1 as(
  select
    customer_id,customer_name,
    date_trunc('month',start_date) as start_month,
    case 
      when contract_term = 'Yearly' then 12
      else 1
    end as no_of_months,
    case 
      when contract_term = 'Yearly' then revenue_amount/12
      else revenue_amount
    end as monthly_revenue
  from `03_gold_catalog`.obt.one_big_table
  where close_status='Won'
  order by customer_id,start_date
),
cte2 as(
  select customer_id,customer_name,
  monthly_revenue,
  explode(sequence(start_month,add_months(start_month, no_of_months - 1), interval 1 month)) as report_month
  from cte1
),
cte3 as (
  select
    customer_id,customer_name,
    report_month,
    count(*) as no_of_products_usage
  from cte2
  group by customer_id ,customer_name,  report_month
),
cte4 as (
  select
    customer_id,customer_name,
    report_month,
    no_of_products_usage,
    lag(no_of_products_usage) over (partition by customer_id order by report_month) as prev_month_usage
  from cte3
),
cte5 as (
  select 
    customer_id,customer_name,
    sum(case when no_of_products_usage > prev_month_usage then 1 else 0 end) as increasing_count,
    sum(case when no_of_products_usage < prev_month_usage then 1 else 0 end) as decreasing_count 
  from cte4
  group by customer_id,customer_name
)
select 
  customer_id,customer_name,
  case 
    when increasing_count > decreasing_count then 'Expanding'
  end
from cte5 
where increasing_count > decreasing_count;

In [0]:
-- 5
create or replace table `03_gold_catalog`.kpis_from_obt.kpi5 as
with cte1 as(
  select
    customer_id,customer_name,
    date_trunc('month',start_date) as start_month,
    case 
      when contract_term = 'Yearly' then 12
      else 1
    end as no_of_months,
    case 
      when contract_term = 'Yearly' then revenue_amount/12
      else revenue_amount
    end as monthly_revenue
  from `03_gold_catalog`.obt.one_big_table
  where close_status='Won'
  order by customer_id,start_date
),
cte2 as(
  select customer_id,customer_name,
  monthly_revenue,
  explode(sequence(start_month,add_months(start_month, no_of_months - 1), interval 1 month)) as report_month
  from cte1
),
cte3 as (
  select
    customer_id,customer_name,
    report_month,
    count(*) as no_of_products_usage
  from cte2
  group by customer_id ,customer_name,  report_month
),
cte4 as (
  select
    customer_id,customer_name,
    report_month,
    no_of_products_usage,
    lag(no_of_products_usage) over (partition by customer_id order by report_month) as prev_month_usage
  from cte3
),
cte5 as (
  select 
    customer_id,customer_name,
    sum(case when no_of_products_usage > prev_month_usage then 1 else 0 end) as increasing_count,
    sum(case when no_of_products_usage < prev_month_usage then 1 else 0 end) as decreasing_count 
  from cte4
  group by customer_id,customer_name
)
select 
  customer_id,customer_name,
  case 
    when increasing_count < decreasing_count then 'Reducing'
  end
from cte5 
where increasing_count < decreasing_count;

In [0]:
-- 6
create or replace table `03_gold_catalog`.kpis_from_obt.kpi6 as
with cte1 as(
  select
    customer_id,customer_name,
    start_date,
    end_date,
    min(start_date) over(partition by customer_id) as first_start_date,
    date_trunc('month',start_date) as start_month,
    case 
      when contract_term = 'Yearly' then 12
      else 1
    end as no_of_months,
    case 
      when contract_term = 'Yearly' then revenue_amount/12
      else revenue_amount
    end as monthly_revenue
  from `03_gold_catalog`.obt.one_big_table
  where close_status='Won'
  order by customer_id,start_date
),
cte2 as(
  select customer_id,customer_name,
  start_date,
  end_date,
  first_start_date,
  monthly_revenue,
  explode(sequence(start_month,add_months(start_month, no_of_months - 1), interval 1 month)) as report_month
  from cte1
  where first_start_date < "2024-04-01"
),
cte3 as (
  select *
  from cte2
  where 
    first_start_date < "2024-04-01" and
    start_date < "2024-04-01" and 
    report_month >= "2024-04-01" and
    report_month < "2025-04-01"
)
select customer_id,customer_name,
  sum(monthly_revenue) as retained_revenue
from cte3
group by customer_id,customer_name

In [0]:
-- 7
create or replace table `03_gold_catalog`.kpis_from_obt.kpi7 as
with cte1 as(
  select
    customer_id,
    start_date,
    min(start_date) over(partition by customer_id ) as first_start_date,
    date_trunc('month',start_date) as start_month,
    case 
      when contract_term = 'Yearly' then 12
      else 1
    end as no_of_months,
    case 
      when contract_term = 'Yearly' then revenue_amount/12
      else revenue_amount
    end as monthly_revenue
  from `03_gold_catalog`.obt.one_big_table
  where close_status='Won'
  order by customer_id,start_date
),
cte2 as(
  select customer_id,
  start_date,
  first_start_date,
  monthly_revenue,
  explode(sequence(start_month,add_months(start_month, no_of_months - 1), interval 1 month)) as report_month
  from cte1
  where first_start_date < "2024-04-01"
  and start_date < "2024-04-01"
),
cte3 as (
  select customer_id,
    monthly_revenue,
    case 
      when report_month >= "2023-04-01" and report_month < "2024-04-01" then "prev_fy"
      when report_month >= "2024-04-01" and report_month < "2025-04-01" then "cur_fy"
    end as financial_year
  from cte2
  where report_month >= "2023-04-01" and report_month < "2025-04-01"
),
cte4 as (
  select
    customer_id,
    sum(case when financial_year="prev_fy" then monthly_revenue else 0 end) as prev_fy_revenue,
    sum(case when financial_year="cur_fy" then monthly_revenue else 0 end) as cur_fy_revenue
  from cte3
  group by customer_id
)
select 
  sum(case when cur_fy_revenue > prev_fy_revenue then cur_fy_revenue-prev_fy_revenue else 0 end) as upgrade_revenue,
  sum(case when prev_fy_revenue > cur_fy_revenue then prev_fy_revenue-cur_fy_revenue else 0 end) as loss_revenue
from cte4

In [0]:
-- 8
create or replace table `03_gold_catalog`.kpis_from_obt.kpi8 as
with params as (
  select date("2024-07-01") as cur_date
),
params2 as (
  select cur_date,
  add_months(date_trunc('year',add_months(cur_date,-3)),3) as cur_fy_start
  from params
),
params3 as (
  select *,
    add_months(cur_fy_start,-12) as prev_fy_start,
    add_months(cur_date, -12) as prev_year_cur_date
  from params2
),
cte1 as(
  select
    customer_id,
    start_date,
    min(start_date) over(partition by customer_id ) as first_start_date,
    date_trunc('month',start_date) as start_month,
    case 
      when contract_term = 'Yearly' then 12
      else 1
    end as no_of_months,
    case 
      when contract_term = 'Yearly' then revenue_amount/12
      else revenue_amount
    end as monthly_revenue
  from `03_gold_catalog`.obt.one_big_table
  where close_status='Won'
  order by customer_id,start_date
),
cte2 as(
  select 
  monthly_revenue,
  explode(sequence(start_month,add_months(start_month, no_of_months - 1), interval 1 month)) as report_month
  from cte1
),
cte3 as (
select
  monthly_revenue,
  case
    when report_month >= prev_fy_start and report_month < prev_year_cur_date then "prev_fy"
    when report_month >= cur_fy_start and report_month < cur_date then "cur_fy"
  end as fy_year
from cte2 c
cross join params3 p
where report_month >= prev_fy_start and report_month < cur_date
),
cte4 as (
  select
    sum(case when fy_year="prev_fy" then monthly_revenue else 0 end) as prev_fy_ytd_revenue,
    sum(case when fy_year="cur_fy" then monthly_revenue else 0 end) as cur_fy_ytd_revenue
  from cte3
)
select * from cte4

In [0]:
-- 9
create or replace table `03_gold_catalog`.kpis_from_obt.kpi9 as
with params as (
  select 2024 as fy_year
),

params2 as (
  select 
    fy_year,
    make_date(fy_year,4,1) as fy_start,
    make_date(fy_year+1,4,1) as fy_end
  from params
),
cte1 as (
  select
    customer_id,customer_name,
    date_trunc('month', start_date) as start_month,
    case 
      when contract_term = 'Yearly' then 12
      else 1
    end as no_of_months,
    case 
      when contract_term = 'Yearly' then revenue_amount / 12
      else revenue_amount
    end as monthly_revenue
  from `03_gold_catalog`.obt.one_big_table
  where close_status = 'Won'
),
cte2 as (
  select
    customer_id,customer_name,
    monthly_revenue,
    explode(
      sequence(
        start_month,
        add_months(start_month, no_of_months - 1),
        interval 1 month
      )
    ) as report_month
  from cte1
),
cte3 as (
  select
    c.customer_id,c.customer_name,
    c.monthly_revenue
  from cte2 c
  cross join params2 p
  where 
    c.report_month >= p.fy_start and c.report_month <  p.fy_end
),
cte4 as (
  select
    customer_id,customer_name,
    sum(monthly_revenue) as total_revenue
  from cte3
  group by customer_id,customer_name
)
select 
  customer_id,customer_name,
  total_revenue,
  rank() over(order by total_revenue desc) as rnk
from cte4 
order by total_revenue desc;

In [0]:
-- 10
create or replace table `03_gold_catalog`.kpis_from_obt.kpi10 as
with cte1 as (
  select
    customer_id,
    date_trunc('month', start_date) as start_month,
    case 
      when contract_term = 'Yearly' then 12
      else 1
    end as no_of_months,
    case 
      when contract_term = 'Yearly' then revenue_amount / 12
      else revenue_amount
    end as monthly_revenue
  from `03_gold_catalog`.obt.one_big_table
  where close_status = 'Won'
),
cte2 as (
  select
    customer_id,
    monthly_revenue,
    explode(
      sequence(
        start_month,
        add_months(start_month, no_of_months - 1),
        interval 1 month
      )
    ) as report_month
  from cte1
),
cte3 as(
  select
    report_month,
    sum(monthly_revenue) as total_monthly_revenue
  from cte2
  group by report_month
)
select
  year(report_month) as year,
  month(report_month) as month,
  sum(total_monthly_revenue) over(order by report_month rows between 11 preceding and current row) as rolling_12m_revenue
from cte3
order by report_month;